# GPU Setup

In [64]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("device:", device)

if torch.cuda.is_available():
    print("GPU: ",torch.cuda.get_device_name(0))

torch.manual_seed(42)

device: cuda
GPU:  Tesla T4


# Sample Restaurant Corpus 1

In [65]:
sentences = [
    "i want chinese food",
    "i want korean food",
    "i want italian food",
    "i want japanese food",
    "i want thai food",
    "i want indian food",
    "i want a chinese restaurant",
    "i want a korean restaurant",
    "i want an italian restaurant",
    "i want a japanese restaurant",
    "i want to eat chinese food",
    "i want to eat korean food",
    "i want to eat italian food",
    "i want to eat japanese food",
    "i would like chinese food",
    "i would like korean food",
    "i would like italian food",
    "i would like japanese food",
    "find a chinese restaurant",
    "find a korean restaurant",
    "find an italian restaurant",
    "find a japanese restaurant",
    "show me a chinese restaurant",
    "show me a korean restaurant",
    "show me an italian restaurant",
    "show me a japanese restaurant",
    "can you find a chinese restaurant",
    "can you find a korean restaurant",
    "can you find an italian restaurant",
    "can you find a japanese restaurant",
    "where can i eat chinese food",
    "where can i eat korean food",
    "where can i eat italian food",
    "where can i eat japanese food",
    "i want cheap chinese food",
    "i want cheap korean food",
    "i want expensive italian food",
    "find a cheap restaurant",
    "find a good restaurant",
    "find a nearby restaurant",
    "i want a restaurant downtown",
    "i want a restaurant nearby",
    "show me a restaurant downtown",
    "the chinese restaurant is good",
    "the korean restaurant is good",
    "the italian restaurant is expensive",
    "the thai restaurant is cheap",
]

# Build the Voca


In [66]:
# pad : 짧은 문장 보정, bos : 문장 시작, eos : 문장의 끝, unk : 모르는 단어
SPECIAL_TOKENS = [
    "<pad>",
    "<bos>",
    "<eos>",
    "<unk>"
]

words = []

for sentence in sentences:
    words.extend(
            sentence.lower().split()
    )

vocab = SPECIAL_TOKENS + sorted(
    set(words)
)

word_to_id = {
    word: i
    for i, word in enumerate(vocab)
}

id_to_word = {
    i: word
    for word, i in word_to_id.items()
}


# 상수로 정의 해주고요
PAD_ID = word_to_id["<pad>"]
BOS_ID = word_to_id["<bos>"]
EOS_ID = word_to_id["<eos>"]
UNK_ID = word_to_id["<unk>"]

vocab_size = len(vocab)
print("Vocabulary size:", vocab_size)
print(word_to_id)

Vocabulary size: 33
{'<pad>': 0, '<bos>': 1, '<eos>': 2, '<unk>': 3, 'a': 4, 'an': 5, 'can': 6, 'cheap': 7, 'chinese': 8, 'downtown': 9, 'eat': 10, 'expensive': 11, 'find': 12, 'food': 13, 'good': 14, 'i': 15, 'indian': 16, 'is': 17, 'italian': 18, 'japanese': 19, 'korean': 20, 'like': 21, 'me': 22, 'nearby': 23, 'restaurant': 24, 'show': 25, 'thai': 26, 'the': 27, 'to': 28, 'want': 29, 'where': 30, 'would': 31, 'you': 32}


# Prepare Next-word Training Data
다음 단어 예측용 학습 데이터 만들기

In [67]:
MAX_LEN = 12

class LanguageModelDataset(Dataset):
    def __init__(self, sentences):
        self.examples = []

        for sentence in sentences:
            # 토큰화
            tokens = sentence.lower().split()

            # ID로 변환하기
            ids = [BOS_ID] # 문장 시작 표시
            for word in tokens:
                # 모르는 거는 모른다고 번호 입력
                ids.append(word_to_id.get(word,UNK_ID))
            ids.append(EOS_ID) # 문장 종료 표시

            # 입력
            x = ids[:-1] # BOS_ID 부터 EOS_ID 전까지 EOS_ID는 버림
            # 다음 단어 타겟
            y = ids[1:] # x[t]를 보고 y[t](= 다음 단어)를 맞히도록 짝을 맞춤

            # 긴거는 자르고 짧은거는 패딩을 붙이고
            x = x[:MAX_LEN]
            y = y[:MAX_LEN]
            padding = MAX_LEN - len(x)
            x = x + [PAD_ID] * padding
            y = y + [PAD_ID] * padding

            # 텐서로 변환하기
            self.examples.append(
                (
                    torch.tensor( x, dtype=torch.long ),
                    torch.tensor( y, dtype=torch.long )
                )
            )

    # 샘플 개수 반환 함수
    def __len__(self):
        return len(self.examples)

    # idx번째 (x, y) 쌍 꺼내기
    def __getitem__(self, index):
        return self.examples[index]

dataset = LanguageModelDataset( sentences )

# loader의 역할
# 16개의 문장 배치로 묵기
# 문장 섞기
loader = DataLoader(
    dataset,
    batch_size = 16,
    shuffle=True
)

x, y = dataset[0]
print("Input IDs :", x)
print("Target IDs :", y)

Input IDs : tensor([ 1, 15, 29,  8, 13,  0,  0,  0,  0,  0,  0,  0])
Target IDs : tensor([15, 29,  8, 13,  2,  0,  0,  0,  0,  0,  0,  0])


# Model Training
[학습 1스텝 순서]
1. GPU로 옮기기        : x.to(device), y.to(device)
2. 이전 배치 기울기 초기화 : optimizer.zero_grad()
3. 순전파             : logits = model(x)
4. 로스 구하기         : loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
5. 역전파             : loss.backward()  → 각 파라미터의 기울기 계산
6. 파라미터 업데이트     : optimizer.step() → 5번에서 구한 기울기로 AdamW가 가중치 수정


In [68]:
# Model Training 1

def train_model(
        model,
        loader,
        epochs=200,
        learning_rate=0.003
):

    model = model.to(device)

    # loss 구하기, PAD_ID는 무시
    criterion = nn.CrossEntropyLoss(
        ignore_index=PAD_ID
    )

    # optimizer 구하기 , AdamW 사용하기
    optimizer = torch.optim.AdamW(
        # 모델의 모든 파라미터를 대상으로 진행하겠다는 의미
        model.parameters(),
        lr=learning_rate
    )

    model.train()

    # Model Training 2

    for epoch in range(epochs):
        total_loss = 0

        for x, y in loader:
            # GPU로 옮기기
            x = x.to(device)
            y = y.to(device)

            # 이전 배치 기울기 초기화
            optimizer.zero_grad()

            # 순전파
            logits = model(x)

            # loss 구하기, CrossEntropyLoss이용
            loss = criterion(
                logits.reshape(-1, vocab_size), y.reshape(-1),
            )

            loss.backward()  # 역전파: 기울기 계산
            optimizer.step() # AdamW로 파라미터 업데이트

            total_loss += loss.item()          # 텐서 → 파이썬 float

        if (epoch + 1) % 20 == 0:
            average_loss = total_loss / len(loader)   # len(loader) = 배치 개수(3)
            print(f"Epoch: {epoch+1:3d},  Loss: {average_loss:.4f}")





## AdamW 공부

### Adam 이란? - (Adaptive Moment Estimation)

- Adam 옵티마이저는 딥러닝 모델 학습에 널리 사용되는 최적화 알고리즘
- Stochastic Gradient Descent(SGD, 확률적 경사하강법)의 확장 기법인 **Momentum**(1차 모멘트)과 **RMSProp**(2차 모멘트)의 장점을 결합
  - 기울기의 1차, 2차 모멘트 추정치로 **파라미터마다 학습률을 개별적으로 조절** → 기존 방식보다 빠르고 안정적으로 수렴
  


- **Momentum**: 이전 기울기 방향을 기억해서 관성을 줌 → 진동이 줄고 빨리 내려감
  - 비유: 언덕을 굴러가는 볼링공. 작은 요철에 흔들리지 않음

- **AdaGrad**: 지금까지 기울기 제곱을 **전부 누적**해서, 많이 움직인 파라미터는 보폭을 줄임
  - 문제: 누적만 하니까 G가 계속 커져서 **학습률이 0에 가까워지고 학습이 멈춤**

- **RMSProp**: AdaGrad의 "전부 누적"을 **최근 위주 이동평균**으로 바꿈 → 오래된 기울기는 서서히 잊음
  - 학습률이 0으로 죽지 않음

- **Adam**: Momentum의 m(방향) + RMSProp의 v(보폭)
  - $m = \beta_1 m + (1-\beta_1) g$ → **1차 모멘트** = 기울기의 이동평균 (어느 방향으로 갈지)
  - $v = \beta_2 v + (1-\beta_2) g^2$ → **2차 모멘트** = 기울기 제곱의 이동평균 (얼마나 흔들리는지)
  - $w = w - lr \cdot \frac{\hat{m}}{\sqrt{\hat{v}}+\epsilon}$
  - $\hat{m}, \hat{v}$: 초반에 m, v가 0에서 시작해 작게 나오는 걸 보정한 값 (bias correction)
  - $\epsilon$: 분모가 0이 되는 것을 막는 아주 작은 수 (보통 $10^{-8}$)
  - 기본값: $lr = 0.001$, $\beta_1 = 0.9$, $\beta_2 = 0.999$


---
비유로 알아보자

| 옵티마이저 | 등산객 |
|---|---|
| SGD | 발밑 경사만 보고 한 걸음씩 |
| Momentum | 내려오던 관성을 살려서 성큼성큼 |
| AdaGrad | 걸어온 총 거리만큼 지쳐서 보폭이 줄다가 결국 멈춤 |
| RMSProp | 최근 지형만 보고 보폭 조절 → 지치지 않음 |
| Adam | 관성 + 최근 지형 보고 보폭 조절 |
| AdamW | Adam + 배낭 무게(가중치)를 매번 일정 비율로 덜어냄 |

---
### AdamW 란? - (Adam with decoupled Weight decay)

- Adam에 **Weight Decay를 "분리(decoupling)"해서** 적용한 옵티마이저
- 현재 LLM 학습의 표준 (GPT, Llama 등)

#### 먼저, Weight Decay란?
- 가중치가 너무 커지지 않게 **매 스텝 조금씩 0 쪽으로 줄이는 것**
- 목적: 가중치가 커지면 학습 데이터를 외워 버림(과적합) -> 이를 방지
  - $w = w - lr \cdot \lambda \cdot w$ ($\lambda$ = weight decay 계수)
  - 비유: 화분 가지치기. 가지(가중치)가 제멋대로 뻗지 않게 매번 다듬기

#### 기존 Adam의 문제 (L2 Regularization 방식)
- Weight decay를 **손실 함수에 직접 더해서** 구현함
  - 손실 = 원래 손실 + $\frac{\lambda}{2} w^2$ → 기울기 = $g + \lambda w$
- 그런데 Adam은 기울기를 $\sqrt{v}$로 나눔 → **decay 항까지 같이 나눠져 버림**
  - 과거 기울기가 컸던 파라미터 ($\sqrt{v}$ 큼) → decay가 **약해짐**
  - 과거 기울기가 작았던 파라미터 ($\sqrt{v}$ 작음) → decay가 **세짐**
- 결과: 같은 $\lambda$를 줬는데 **파라미터마다 규제 강도가 제멋대로 달라짐**

#### AdamW의 해결
- Weight decay를 기울기 계산에서 **분리**해서 **맨 마지막에 따로** 적용
  1. Adam 업데이트는 **순수한 손실 기울기만으로** 계산
     - $w = w - lr \cdot \frac{\hat{m}}{\sqrt{\hat{v}}+\epsilon}$
  2. 그다음 weight decay를 **가중치에서 직접 빼기** ($\sqrt{v}$로 안 나눔!)
     - $w = w - lr \cdot \lambda \cdot w$
- 결과: 모든 파라미터가 **자기 크기에 비례해서 같은 비율로** 줄어듦

#### 비유: 월급과 세금
- Adam + L2: 세금을 소득 계산에 섞은 뒤 사람마다 다른 보정률($\sqrt{v}$)로 나눔 → 고소득자일수록 세금이 깎이는 불공평한 세제
- AdamW: 소득 보정은 보정대로, 세금은 **모든 재산에 같은 세율로 따로** 걷음 → 공평

#### 코드
```python
optimizer = torch.optim.AdamW(
    model.parameters(),    # 모델의 모든 파라미터(W, b)를 대상으로
    lr=learning_rate,      # 학습률 (보폭)
    weight_decay=0.01      # 생략 시 기본값 0.01 (Adam은 기본값 0)
)
```

- 실무: LLM은 보통 `weight_decay=0.1`, `betas=(0.9, 0.95)` 사용

# Text Generation
학습이 끝난 모델로 실제 문장을 만드는 것!

[텍스트 생성 순서]

0. 준비      : model.eval() + torch.no_grad() → 평가 모드, 기울기 계산 OFF (학습 안 하니까)
1. 인코딩    : "i want" → [<bos>, i, want] → [1, 15, 29]  (없는 단어는 <unk>)
2. 문맥 자르기 : context = ids[-MAX_LEN:] → 학습 때처럼 최근 12개만 사용
3. 순전파    : logits = model(x) → [1, 문맥길이, 33] 모든 위치의 다음 단어 점수
4. 마지막만   : logits[0, -1] → 맨 끝 위치 점수만 사용 (= 지금 문장 다음에 올 단어)
5. 온도 조절  : / temperature → 낮으면 뾰족(안전), 높으면 평평(다양)
6. 후보 추리기 : top-k → 점수 상위 k개만 남김
7. 추첨      : softmax로 확률 변환 → multinomial로 확률 비례 1개 뽑기
8. 종료 확인  : <eos>면 break, 아니면 ids에 붙이고 2번으로 반복
9. 디코딩    : ID → 단어, 특수 토큰 제거, " ".join으로 문장 완성

In [69]:
# Text Generation

def generate_text(
        model, prompt="i want",
        max_new_tokens=12,
        temperature=0.8,
        top_k=3
):
    model.eval() # 평가 모드
    # 1. 인코딩 : 프롬프트 → 토큰 ID 리스트  ("i want" -> [<bos>, i, want] = [1, 15, 29])
    prompt_words = (
        prompt.lower().split()
        if prompt
        else []
    )
    ids = [BOS_ID]
    for word in prompt_words:
        ids.append(word_to_id.get(word, UNK_ID))   # 단어장에 없으면 <unk>

    # 기울기 계산 off
    with torch.no_grad():

        for _ in range(max_new_tokens): # 최대 12개 단어까지 새로 생성
            # 2. 문맥 자르기 최근 MAX_LEN개 토큰만 문맥으로 사용
            context = ids[-MAX_LEN:]
            x = torch.tensor([context], dtype=torch.long, device=device)

            # 3. 순전파 -> 모든 마지막(문장의 마지막 단어) 위치의 다음 단어 점수
            logits = model(x)

            # 4. 마지막 위치의 점수만 필요 (= 지금까지 문장 다음에 올 단어)
            next_logits = logits[0, -1]
            # 5. 온도 조절
            next_logits = next_logits / temperature
            # temperature < 1 : 점수 차이가 커짐 -> 확률이 뾰족 -> 안전, 예측 가능한 문장, 비슷한 문장
            # temperature > 1 : 점수 차이가 줄어듦 -> 확률이 평평 -> 다양, 무작위, 다양성 높고 창의적인 문장
            # T     0.3       5
            # eat	0.74	0.23
            # drink	0.19	0.21
            # have	0.05	0.20
            # order	0.01	0.18
            # get	0.00    0.17
            # 이렇게 되면서 다음에 올 단어가 선정되는 데 영향을 미치는 것!

            # 6. Top-k 샘플링 : 점수 상위 k개 후보 중에서만 추첨
            if top_k is not None:
                values, indices = torch.topk(next_logits, top_k)   # 상위 k개 점수와 그 단어 ID

                # 7. 확률 기반 추첨 진행!
                probabilities = F.softmax(values, dim=-1)          # k개 점수 → 합이 1인 확률
                sampled_index = torch.multinomial(probabilities, 1)  # 확률에 비례해 1개 추첨 (rand)
                next_id = indices[sampled_index].item()            # 후보 내 순번 → 실제 단어 ID
            else: # top_k=None이면 전체 단어에서 추첨
                probabilities = F.softmax(next_logits, dim=-1)
                next_id = torch.multinomial(probabilities, 1).item()

            # 8. <eos>가 나오면 문장 끝 → 생성 중단
            if next_id == EOS_ID:
                break

            ids.append(next_id) # 뽑은 단어를 이어 붙이고 다시 2번 단계로 이동 (autoregressive)

    # 9. ID → 단어로 바꾸면서 특수 토큰(<bos> 등)은 제거
    generated_words = []
    for token_id in ids:
        word = id_to_word[token_id]
        if word not in SPECIAL_TOKENS:
            generated_words.append(word)

    return " ".join(generated_words)

# RNN Language Model
- RNN 모델 생성 단계

In [70]:
class RNNLanguageModel(nn.Module):

    def __init__(
            self, vocab_size,
            embedding_dim = 64,
            hidden_dim=128
    ):

        # 64차원 벡터 (Word2Vec의 임베딩과 같은 lookup 테이블)
        super().__init__()
        self.embedding = nn.Embedding(
                vocab_size, embedding_dim,
                padding_idx=PAD_ID
        )

        # hidden state를 갱신
        self.rnn = nn.RNN(input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True )

        # hidden state -> 단어장 33개 단어 각각의 점수 ( logits )
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # 입력 Xt
        embedded = self.embedding(x)

        # hidden state ht
        hidden_states , hidden = self.rnn(embedded)

        # 출력 Yt
        logits = self.output(hidden_states)

        return logits

# RNN 학습과 생성
rnn_model = RNNLanguageModel(vocab_size).to(device)
print(rnn_model)
train_model(rnn_model, loader)

print("\n[RNN 생성 결과]")
for _ in range(5):
    print(generate_text(rnn_model, prompt="i want"))

RNNLanguageModel(
  (embedding): Embedding(33, 64, padding_idx=0)
  (rnn): RNN(64, 128, batch_first=True)
  (output): Linear(in_features=128, out_features=33, bias=True)
)
Epoch:  20,  Loss: 0.6952
Epoch:  40,  Loss: 0.6670
Epoch:  60,  Loss: 0.6598
Epoch:  80,  Loss: 0.6603
Epoch: 100,  Loss: 0.6585
Epoch: 120,  Loss: 0.6568
Epoch: 140,  Loss: 0.6534
Epoch: 160,  Loss: 0.6548
Epoch: 180,  Loss: 0.6536
Epoch: 200,  Loss: 0.6542

[RNN 생성 결과]
i want a restaurant downtown
i want to eat korean food
i want a restaurant nearby
i want cheap korean food
i want to eat chinese food


- Trace one sentence through training and generation

Given one sentence: i want chinese food

Make a function : trace_model(model, sentence)

The function can print out the following
     Original sentence
     Token IDs
     Training input x
     Training target y
     Embedding/input shape
     Model logits shape
     Last-position logits shape
     Top-5 predicted words


You should use the RNNLanguageModel class in Lab4_RNN.

The class will be covered in Lecture later on.

You just use this class for this homework.

In [71]:
def trace_model(model, sentence):

    # 0. 평가 모드에서 진행
    model.eval()

    # Original Sentence 출력하기
    print("Original Sentence :", sentence)

    # 1. 인코딩 : 문장 → 토큰 ID 리스트 ("i want chinese food" -> [<bos>, i, want, chinese, food, <eos>] = [1, 15, 29, 8, 13, 2])  # ✏️ 들여쓰기 4칸으로
    sentence_words = (
        sentence.lower().split()
        if sentence
        else []
    )
    ids = [BOS_ID]
    for word in sentence_words:
        ids.append(word_to_id.get(word, UNK_ID))   # 단어장에 없으면 <unk>
    ids.append(EOS_ID) # 문장 종료 표시

    # Token IDs 출력하기
    print("Token IDs :", ids)

    # Training input x - 마지막 하나(<eos>) 뺀 것
    x_ids = ids[:-1]
    print("Training input x :", x_ids, [id_to_word[i] for i in x_ids])

    # Training target y - 처음 하나(<bos>) 뺀 것
    y_ids = ids[1:]
    print("Training target y :", y_ids, [id_to_word[i] for i in y_ids])

    # x_ids → 텐서 ([ ]로 한 번 더 감싸서 batch 차원 추가 → [1, 5])
    x = torch.tensor([x_ids], dtype=torch.long, device=device)

    # 기울기 계산 off
    with torch.no_grad():
        embedded = model.embedding(x)   # = RNN 입력 x_t (단어 ID → 64차원 벡터) -> shape 확인용
        logits = model(x)               # = RNN 출력 y_t (모든 위치의 다음 단어 점수, softmax 전)

    # Embedding / input shape 출력
    print("Input shape :", tuple(x.shape))             # [batch, seq] = (1, 5) / 문장 1개의 토큰 5개
    print("Embedding shape :", tuple(embedded.shape))  # [batch, seq, 64] = (1, 5, 64) / 문장 1개의 토큰 5개를 각각 64차원 벡터로 표현

    # Model logits shape 출력
    print("Logits shape :", tuple(logits.shape))       # [batch, seq, vocab_size] = (1, 5, 33) / 토큰 5개 위치마다 단어 33개의 점수

    # 마지막 위치(마지막 단어 다음)의 점수만 꺼내기  # food ->  마지막 단어
    last_logits = logits[0, -1]
    print("Last-position logits shape :", tuple(last_logits.shape))   # (33,) / 마지막 단어 뒤에 올 단어 33개의 점수

    # Top-5 : 점수 -> 확률(softmax) -> 상위 5개
    probabilities = F.softmax(last_logits, dim=-1)        # 33개 점수 → 합이 1인 확률
    top_probs, top_ids = torch.topk(probabilities, 5)     # 확률 상위 5개의 (값, 위치=단어 ID)

    print("Top-5 predicted words :")
    for k in range(5):
        word_id = top_ids[k].item()     # k번째 단어 ID
        prob = top_probs[k].item()      # k번째 확률
        print(f"  {k + 1}. {id_to_word[word_id]:<10} {prob:.4f}")   # 순위, 단어, 확률(소수 4자리)로 정리


trace_model(rnn_model, "i want")

Original Sentence : i want
Token IDs : [1, 15, 29, 2]
Training input x : [1, 15, 29] ['<bos>', 'i', 'want']
Training target y : [15, 29, 2] ['i', 'want', '<eos>']
Input shape : (1, 3)
Embedding shape : (1, 3, 64)
Logits shape : (1, 3, 33)
Last-position logits shape : (33,)
Top-5 predicted words :
  1. a          0.2571
  2. to         0.2201
  3. cheap      0.1117
  4. an         0.0543
  5. korean     0.0537
